<a href="https://colab.research.google.com/github/max7082/colab-practice/blob/main/%EC%9C%A4%ED%98%84%EC%9B%85_%EA%B5%AC%ED%98%84.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import torch
from torch import nn
from torchvision.datasets import MNIST

device = "cuda" if torch.cuda.is_available() else "cpu"

data = MNIST("./data", train=True, download=True)
test_data = MNIST("./data", train=False, download=True)

x = data.data[:2000].float().reshape(2000, 784).to(device) / 255
label = data.targets[:2000].to(device)
y = nn.functional.one_hot(label, num_classes=10).float()

test_x = test_data.data.float().reshape(10000, 784).to(device) / 255
test_y = test_data.targets.to(device)

for method in ["ERM", "입력만 혼합", "정답만 혼합", "mixup"]:
    torch.manual_seed(42)

    model = nn.Sequential(
        nn.Linear(784, 64),
        nn.ReLU(),
        nn.Linear(64, 10)
    ).to(device)

    optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

    for _ in range(500):
        lam = torch.distributions.Beta(0.4, 0.4).sample().item()
        j = torch.randperm(len(x), device=device)

        mixed_x = lam * x + (1 - lam) * x[j]
        mixed_y = lam * y + (1 - lam) * y[j]

        if method == "ERM":
            input_x = x
            target_y = y
        elif method == "입력만 혼합":
            input_x = mixed_x
            target_y = y if lam >= 0.5 else y[j]
        elif method == "정답만 혼합":
            input_x = x
            target_y = mixed_y
        else:
            input_x = mixed_x
            target_y = mixed_y

        loss = nn.functional.cross_entropy(model(input_x), target_y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        train_prediction = model(x).argmax(1)
        train_accuracy = (
            (train_prediction == label).float().mean().item()
        )

        test_prediction = model(test_x).argmax(1)
        test_accuracy = (
            (test_prediction == test_y).float().mean().item()
        )

    print(
        f"{method}: "
        f"train accuracy {train_accuracy:.3%}, "
        f"test accuracy {test_accuracy:.3%}"
    )

ERM: train accuracy 95.000%, test accuracy 89.300%
입력만 혼합: train accuracy 93.600%, test accuracy 88.660%
정답만 혼합: train accuracy 91.400%, test accuracy 87.500%
mixup: train accuracy 93.350%, test accuracy 88.340%
